In [ ]:
# Project 8 - Decision trees and neural networks

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report,confusion_matrix
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
df = pd.read_csv('data/zombies.csv')
df.head()


In [ ]:
plt.hist(df['zombies_killed'], bins=30, color='skyblue', edgecolor='black')

# Add labels and title
plt.title('Basic Histogram')
plt.xlabel('Values')
plt.ylabel('Frequency')

# Display the plot
plt.show()

In [ ]:
# Calculate Body Mass Index (height in inches, weight in lbs)
bmi = (df['weight'] / (df['height'] ** 2)) * 703

# Logit equation combining stamina, combat record, and physical vulnerability
# High kills & education help; very high/low age and extreme BMI hurt
logit = (
    0.7 * df['zombies_killed'] 
    + 0.15 * df['years_of_education'] 
    - 0.05 * np.abs(df['age'] - 28) 
    - 0.10 * np.abs(bmi - 22) 
    - 1.5
)

# Convert logit to probability and sample binary outcome
prob = 1 / (1 + np.exp(-logit))
np.random.seed(42)
df['survived'] = np.random.binomial(1, prob)


df.head()

In [ ]:
df['major'].unique()

In [ ]:
plt.scatter(df['age'], df['weight'])

In [ ]:
df_hot_encoded = pd.get_dummies(df, columns=['major', 'gender'], dtype=int)
df_hot_encoded.head()

In [ ]:
X = df_hot_encoded.drop(['survived', 'first_name', 'last_name', 'id'], axis=1)
y = df_hot_encoded['survived']

X_train, X_test, y_train, y_test = train_test_split(X,y) 

In [ ]:
print(X.columns)
print(y.unique())

In [ ]:
scaler = StandardScaler()

scaler.fit(X_train)

X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

X_train


## Neural Net Version 1

In [ ]:
mlp_1 = MLPClassifier(hidden_layer_sizes=(10,10,15,20,30,8), max_iter=5000)

In [ ]:
mlp_1.fit(X_train, y_train)

In [ ]:
predictions_1 = mlp_1.predict(X_test)

In [ ]:
print(confusion_matrix(y_test, predictions_1))

In [ ]:
print(classification_report(y_test, predictions_1))

In [ ]:
import networkx as nx
import colorsys

def show_ann(mlp):
    hidden_layers_n = len(mlp.coefs_)-1
    layers_n = hidden_layers_n + 2
    input_neurons_n = len(mlp.coefs_[0])
    hidden_neurons_n = [len(mlp.coefs_[i+1]) for i in range(hidden_layers_n)]
    output_neurons_n = len(mlp.coefs_[-1][0])

    G = nx.DiGraph()
    pos = {}

    # Create the neurons of the input layer
    for i in range(input_neurons_n):
        pos['Layer0_{}'.format(i)] = (i,layers_n-1)

    for j in range(hidden_layers_n):
        # Create the neurons of the j'th hidden layer
        prev_layer = j
        cur_layer = j+1
        if (j == 0):
            prev_size = input_neurons_n
        else:
            prev_size = hidden_neurons_n[j-1]
        for i in range(hidden_neurons_n[j]):
            pos['Layer{}_{}'.format(cur_layer,i)] = (i,layers_n-1-cur_layer)
            for k in range(prev_size):
                w = mlp.coefs_[prev_layer][k][i]
                G.add_edge('Layer{}_{}'.format(prev_layer,k),'Layer{}_{}'.format(cur_layer,i), weight=w)

    # Create the neurons of the output layer
    prev_layer = hidden_layers_n
    cur_layer = hidden_layers_n+1
    for i in range(output_neurons_n):
        pos['Layer{}_{}'.format(cur_layer,i)] = (i,layers_n-1-cur_layer)
        for k in range(hidden_neurons_n[-1]):
            w = mlp.coefs_[prev_layer][k][i]
            G.add_edge('Layer{}_{}'.format(prev_layer,k),'Layer{}_{}'.format(cur_layer,i), weight=w)

    edges = G.edges()
    colors = [colorsys.hsv_to_rgb(0 if G[u][v]['weight'] < 0 else 0.65,
                                  1,#min(1, abs(G[u][v]['weight'])),
                                  1) for u,v in edges]
    weights = [abs(G[u][v]['weight'])*2 for u,v in edges]

    nx.draw(G, pos, node_color='y', node_size=450, width=weights, edge_color=colors)
    
show_ann(mlp_1)

## Neural Networks Version 2

In [ ]:
mlp_2 = MLPClassifier(hidden_layer_sizes=(15), max_iter=5000)
mlp_2.fit(X_train, y_train)
predictions_2 = mlp_2.predict(X_test)
print(confusion_matrix(y_test, predictions_2))
print(classification_report(y_test, predictions_2))

In [ ]:
show_ann(mlp_2)

## Neural Networks Version 3

In [ ]:
mlp_3 = MLPClassifier(hidden_layer_sizes=(4,5,3,6,7,4,5,2,5,6,7,3,6,5,3), max_iter=5000)
mlp_3.fit(X_train, y_train)
predictions_3 = mlp_3.predict(X_test)
print(confusion_matrix(y_test, predictions_3))
print(classification_report(y_test, predictions_3))

In [ ]:

show_ann(mlp_3)

## Decision Tree

In [ ]:
# For the decision trees you should visualize at least two trees (using code we saw in class) at different maximum depths.
# Discuss the effect the depth has on bias and variance.
# Discuss the choices the algorithm makes in which attributes to split on.
# For example, at a given tree level do all nodes split on the same attribute? Why or why not?

In [ ]:
X = df_hot_encoded.drop(['survived', 'first_name', 'last_name', 'id'], axis=1)
y = df_hot_encoded['survived']

# X_train, X_test, y_train, y_test = train_test_split(X,y) 

### Decision Tree 1

In [ ]:
# fit a classification tree with max_depth=3 on all data
from sklearn.tree import DecisionTreeClassifier
from graphviz import Source
from sklearn import tree
from IPython.display import SVG


treeclf = DecisionTreeClassifier(max_depth=3, random_state=1)
treeclf.fit(X, y)

graph = Source(tree.export_graphviz(treeclf, out_file=None,
                                    feature_names=X.columns,
                                    class_names=['0', '1'], filled = True))
display(SVG(graph.pipe(format='svg')))

In [ ]:
# compute the feature importances
pd.DataFrame({'feature':X.columns, 'importance':treeclf.feature_importances_})

### Decision Tree 2

In [ ]:
treeclf = DecisionTreeClassifier(max_depth=5, random_state=1)
treeclf.fit(X, y)

graph = Source(tree.export_graphviz(treeclf, out_file=None,
                                    feature_names=X.columns,
                                    class_names=['0', '1'], filled = True))
display(SVG(graph.pipe(format='svg')))

In [ ]:
# compute the feature importances
pd.DataFrame({'feature':X.columns, 'importance':treeclf.feature_importances_})